# BONUS · B2 · Medallion Architecture: Bronze → Silver → Gold

> **This module is optional.** You can work through it independently after the course or independently after the course if time allows. It builds its own isolated demo tables (prefixed `bonus_b2_`) and cleans up after itself.

## Learning Objectives

By the end of this notebook you can:

- describe the purpose and design principles of each Medallion layer,
- explain what transformations happen at each layer and why,
- identify common data quality problems in raw data and apply Silver-layer fixes,
- build a Bronze → Silver → Gold pipeline for a simple retail dataset,
- explain why Gold is rebuilt from Silver, not from Bronze directly,
- connect Medallion to the RetailHub tables you used throughout this course.

## Business Context

The Gold tables you queried in Day 1 — `gold.fact_sales`, `gold.dim_customer`, `gold.kpi_daily` — were not hand-crafted. They were produced by a pipeline that started with raw operational data, cleaned it through a Silver layer, and aggregated it into Gold.

This notebook builds that pipeline from scratch using a small RetailHub orders dataset, so you can see every transformation decision and why it was made.

## Architecture Overview

```
RAW DATA                  BRONZE                    SILVER                     GOLD
────────────────          ──────────────────────    ──────────────────────     ────────────────────────────
silver.order_lines  ──►  bronze_b2_orders     ──►  silver_b2_orders_clean ──►  gold_b2_daily_revenue
silver.customers    ──►  bronze_b2_customers  ──►  silver_b2_customers    ──►  gold_b2_customer_summary
```

| Layer | Purpose | Write Mode | Quality Enforcement | Consumers |
|---|---|---|---|---|
| **Bronze** | Land raw data as-is + add lineage metadata | Append (facts), Overwrite (dims) | None | Silver pipelines |
| **Silver** | Cleanse, deduplicate, standardize, derive metrics | Overwrite | Hard reject on business keys | Gold pipelines, analysts |
| **Gold** | Business-ready aggregations answering one question each | Overwrite | Assumed clean (inherited from Silver) | BI tools, dashboards, Genie |

**Key principle:** each layer adds value. Bronze = safety net. Silver = trusted entities. Gold = business answers.

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

In [ ]:
required = [
    f"{SILVER}.customers",
    f"{SILVER}.sales_orders",
    f"{SILVER}.order_lines",
]
missing = [t for t in required if not spark.catalog.tableExists(t)]
if missing:
    raise Exception(f"Missing tables: {missing}. Run data/generate_training_dataset.ipynb first.")
print("[OK] Pre-check passed — Silver source tables exist")

## The Problem With Raw Data

Before building anything, explore the Silver tables that the training dataset generator created. These tables intentionally contain the same quality issues you'd find in a real operational system:

| Issue | Where | Why it matters |
|---|---|---|
| Missing `unit_price` | `silver.order_lines` | can't compute revenue without a price |
| Out-of-dictionary `status` | `silver.sales_orders` | dashboards count `Unknown` as valid orders |
| Future-dated orders | `silver.sales_orders` | forecasting metrics become inflated |
| Duplicate `(order_id, product_id)` rows | `silver.order_lines` | double-counts revenue |
| Orphan `customer_id` references | `silver.order_lines` | customer joins silently fail |

In a real project these would live in a **raw Bronze table** before any quality checks. Here we use the Silver tables as the starting point to keep the demo self-contained.

In [ ]:
from pyspark.sql import functions as F

print("=== Data Quality Issues in the Source Data ===")

issues = spark.sql(f"""
    SELECT
        COUNT(*) AS total_lines,
        SUM(CASE WHEN unit_price IS NULL     THEN 1 ELSE 0 END) AS missing_unit_price,
        SUM(CASE WHEN line_revenue IS NULL   THEN 1 ELSE 0 END) AS null_line_revenue
    FROM {SILVER}.order_lines
""")
display(issues)

status_issues = spark.sql(f"""
    SELECT status, COUNT(*) AS order_count
    FROM {SILVER}.sales_orders
    GROUP BY status
    ORDER BY order_count DESC
""")
print("\nOrder status distribution:")
display(status_issues)

future_orders = spark.sql(f"""
    SELECT COUNT(*) AS future_dated_orders
    FROM {SILVER}.sales_orders
    WHERE order_date > current_date()
""")
print("\nFuture-dated orders:")
display(future_orders)

## Layer 1: Bronze — Land Raw Data

> **Bronze rule:** store everything exactly as received. Add lineage metadata. Apply no business logic.

Bronze is the safety net. If Silver processing fails or a business rule changes, engineers can replay the pipeline from Bronze without going back to the source system.

**What Bronze adds:**
- `_load_ts` — when the data was ingested
- `_source` — which system or process produced this batch

**What Bronze does NOT do:**
- filter rows
- fix quality issues
- compute derived columns
- join with other tables

In [ ]:
# Bronze customers: full dimension, overwrite on each refresh
spark.sql(f"""
    CREATE OR REPLACE TABLE {BRONZE}.bonus_b2_customers
    COMMENT 'Bonus B2 demo: Bronze customer dimension — raw copy with lineage metadata only'
    AS
    SELECT
        customer_id,
        customer_name,
        email,
        city,
        region,
        country,
        segment,
        created_date,
        current_timestamp() AS _load_ts,
        'silver.customers'  AS _source
    FROM {SILVER}.customers
""")

print(f"[OK] bronze.bonus_b2_customers: {spark.table(f'{BRONZE}.bonus_b2_customers').count():,} rows")
display(spark.table(f"{BRONZE}.bonus_b2_customers").limit(5))

In [ ]:
# Bronze orders: fact table — append mode preserves history across batches
# (here we use overwrite since we have a single batch)
spark.sql(f"""
    CREATE OR REPLACE TABLE {BRONZE}.bonus_b2_orders
    COMMENT 'Bonus B2 demo: Bronze order lines — raw copy including all quality issues'
    AS
    SELECT
        l.line_id,
        l.order_id,
        l.product_id,
        l.customer_id,
        l.order_date,
        l.channel,
        o.status,
        l.quantity,
        l.unit_price,
        l.unit_cost,
        l.discount_pct,
        l.line_revenue,
        l.line_margin,
        l.category,
        current_timestamp()   AS _load_ts,
        'silver.order_lines'  AS _source
    FROM {SILVER}.order_lines l
    JOIN {SILVER}.sales_orders o ON l.order_id = o.order_id
""")

count = spark.table(f"{BRONZE}.bonus_b2_orders").count()
print(f"[OK] bronze.bonus_b2_orders: {count:,} rows (includes all quality issues)")
display(spark.table(f"{BRONZE}.bonus_b2_orders").limit(5))

### Bronze Layer Summary

| Decision | Choice | Reason |
|---|---|---|
| Schema | Raw columns + `_load_ts` + `_source` | Lineage without transformation |
| Quality filters | None | Bronze captures everything — Silver decides what is valid |
| Write mode (customers) | Overwrite | Dimension = current state, full refresh from source |
| Write mode (orders) | Append in production | Facts are immutable — new batches accumulate over time |
| Format | Delta | ACID, time travel, compaction all available for free |

## Layer 2: Silver — Cleanse and Conform

> **Silver rule:** apply business rules. Validate quality. Standardize formats. One source of truth per entity.

Silver is where raw data becomes trustworthy. The transformations below are the same patterns you would see in any production RetailHub pipeline.

In [ ]:
# Silver customers: deduplication + standardization
spark.sql(f"""
    CREATE OR REPLACE TABLE {SILVER}.bonus_b2_customers
    COMMENT 'Bonus B2 demo: Silver customers — deduplicated, standardized, with processed timestamp'
    AS
    WITH deduplicated AS (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY customer_id
                   ORDER BY _load_ts DESC
               ) AS _rn
        FROM {BRONZE}.bonus_b2_customers
        WHERE customer_id IS NOT NULL       -- hard reject: no business key, no row
    )
    SELECT
        customer_id,
        customer_name,
        LOWER(TRIM(email))        AS email,   -- standardize: consistent format for joins
        city,
        region,
        country,
        segment,
        created_date,
        current_timestamp()       AS _processed_at
    FROM deduplicated
    WHERE _rn = 1                             -- keep latest version only
""")

bronze_count = spark.table(f"{BRONZE}.bonus_b2_customers").count()
silver_count  = spark.table(f"{SILVER}.bonus_b2_customers").count()
print(f"[OK] Bronze: {bronze_count:,} rows → Silver: {silver_count:,} rows (dedup removed {bronze_count - silver_count:,})")
display(spark.table(f"{SILVER}.bonus_b2_customers").limit(5))

In [ ]:
# Silver orders: quality filters + derived business metrics
spark.sql(f"""
    CREATE OR REPLACE TABLE {SILVER}.bonus_b2_orders_clean
    COMMENT 'Bonus B2 demo: Silver orders — quality-filtered, derived revenue/margin columns'
    AS
    SELECT
        line_id,
        order_id,
        product_id,
        customer_id,
        order_date,
        channel,
        status,
        quantity,
        unit_price,
        unit_cost,
        discount_pct,
        line_revenue,
        line_margin,
        category,
        current_timestamp() AS _processed_at
    FROM {BRONZE}.bonus_b2_orders
    WHERE order_id    IS NOT NULL           -- required business key
      AND customer_id IS NOT NULL           -- referential integrity
      AND unit_price  IS NOT NULL           -- cannot compute revenue without price
      AND quantity    > 0                   -- positive quantities only
      AND order_date  <= current_date()     -- reject future-dated records
      AND status      IN ('Completed', 'Returned', 'Cancelled')  -- known statuses only
""")

bronze_count = spark.table(f"{BRONZE}.bonus_b2_orders").count()
silver_count  = spark.table(f"{SILVER}.bonus_b2_orders_clean").count()
print(f"[OK] Bronze: {bronze_count:,} rows → Silver (clean): {silver_count:,} rows (quality filters removed {bronze_count - silver_count:,})")

print("\nQuality check on Silver orders:")
display(spark.sql(f"""
    SELECT
        COUNT(*)                                 AS total_rows,
        SUM(CASE WHEN unit_price IS NULL  THEN 1 ELSE 0 END) AS null_price,
        SUM(CASE WHEN quantity  <= 0      THEN 1 ELSE 0 END) AS bad_quantity,
        SUM(CASE WHEN order_date > current_date() THEN 1 ELSE 0 END) AS future_dated,
        COUNT(DISTINCT status)                   AS distinct_statuses
    FROM {SILVER}.bonus_b2_orders_clean
"""))

### Silver Layer Summary

| Transformation | How | Why |
|---|---|---|
| Null business key reject | `WHERE customer_id IS NOT NULL` | rows without keys are unidentifiable |
| Deduplication | `ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY _load_ts DESC)` | keep latest version of each entity |
| Email standardization | `LOWER(TRIM(email))` | consistent format prevents broken joins |
| Future-date filter | `order_date <= current_date()` | protects reporting metrics from inflated forecasts |
| Status whitelist | `status IN (...)` | unknown statuses are ingestion errors, not business states |
| Missing price filter | `unit_price IS NOT NULL` | revenue cannot be computed — row is not analytically useful |

## Layer 3: Gold — Business-Ready Analytics

> **Gold rule:** answer a specific business question. Optimized for consumption. Never expose raw columns.

Each Gold table answers one question well. A Gold table that tries to answer everything ends up serving no one.

| Business Question | Gold Table | Source |
|---|---|---|
| How does daily revenue trend? | `gold_b2_daily_revenue` | `silver_b2_orders_clean` |
| Who are our most valuable customers? | `gold_b2_customer_summary` | `silver_b2_orders_clean` + `silver_b2_customers` |

In [ ]:
# Gold: daily revenue trend — one row per day
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.bonus_b2_daily_revenue
    COMMENT 'Bonus B2 demo: Gold daily revenue — one row per order_date, Completed orders only'
    AS
    SELECT
        order_date,
        COUNT(DISTINCT order_id)                                        AS order_count,
        SUM(CASE WHEN status = 'Completed' THEN line_revenue ELSE 0 END) AS revenue,
        SUM(CASE WHEN status = 'Completed' THEN line_margin  ELSE 0 END) AS gross_margin,
        ROUND(
            SUM(CASE WHEN status = 'Completed' THEN line_margin ELSE 0 END)
            / NULLIF(SUM(CASE WHEN status = 'Completed' THEN line_revenue ELSE 0 END), 0)
            * 100, 2
        ) AS margin_pct
    FROM {SILVER}.bonus_b2_orders_clean
    GROUP BY order_date
    ORDER BY order_date
""")

result = spark.table(f"{GOLD}.bonus_b2_daily_revenue")
print(f"[OK] gold.bonus_b2_daily_revenue: {result.count()} rows (one per day)")
display(result.orderBy("order_date").limit(10))

In [ ]:
# Gold: customer summary — join Silver orders with Silver customers, then aggregate
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.bonus_b2_customer_summary
    COMMENT 'Bonus B2 demo: Gold customer summary — lifetime metrics per customer'
    AS
    SELECT
        c.customer_id,
        c.customer_name,
        c.city,
        c.region,
        c.segment,
        COUNT(DISTINCT o.order_id)              AS total_orders,
        SUM(CASE WHEN o.status = 'Completed' THEN o.line_revenue ELSE 0 END) AS lifetime_revenue,
        ROUND(AVG(CASE WHEN o.status = 'Completed' THEN o.line_revenue END), 2) AS avg_order_value,
        MAX(o.order_date)                       AS last_order_date
    FROM {SILVER}.bonus_b2_orders_clean o
    JOIN {SILVER}.bonus_b2_customers    c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name, c.city, c.region, c.segment
""")

result = spark.table(f"{GOLD}.bonus_b2_customer_summary")
print(f"[OK] gold.bonus_b2_customer_summary: {result.count():,} rows")
print("\nTop 10 customers by lifetime revenue:")
display(result.orderBy("lifetime_revenue", ascending=False).limit(10))

## Full Pipeline Trace

Row counts tell the story of each transformation:

```
Bronze customers        → Silver (deduped)      → Gold (aggregated)
  10,000 rows                ~10,000 rows             ~10,000 rows (one per customer)

Bronze orders           → Silver (quality QA)   → Gold daily revenue
  360,000 rows              ~335,000 rows              ~730 rows (one per day)
```

**Design decisions that determined the row counts:**
- Bronze = source data volume (no filtering)
- Silver = Bronze minus quality rejects (null keys, bad status, future dates, missing prices)
- Gold = Silver aggregated to business grain (one row per day / one row per customer)

**Connection to the main course:**

The `gold.fact_sales`, `gold.kpi_daily`, and `gold.dim_customer` tables you queried in Day 1 were built using exactly this pattern — Bronze landing, Silver cleaning, Gold aggregation — but for the full RetailHub dataset with 120,000 orders and 360,000 order lines.

In [ ]:
# Compare bonus pipeline revenue with main course Gold
# Note: numbers will differ slightly — the B2 Silver pipeline applies quality filters
# (rejects future-dated orders, missing prices, unknown statuses) that the training
# dataset generator does not apply when building gold.fact_sales directly from silver.
print("=== Comparing bonus pipeline with main course Gold ===")

bonusdaily = spark.sql(f"""
    SELECT ROUND(SUM(revenue), 2) AS total_revenue
    FROM {GOLD}.bonus_b2_daily_revenue
""")

coursedaily = spark.sql(f"""
    SELECT ROUND(SUM(line_revenue), 2) AS total_revenue
    FROM {GOLD}.fact_sales
    WHERE status = 'Completed'
""")

print("Bonus pipeline total revenue (B2 quality-filtered Silver):")
bonusdaily.show()

print("Main course gold.fact_sales (all Completed lines, no quality filter):")
coursedaily.show()

print("The difference reflects rows excluded by the B2 Silver quality gates")
print("(future-dated orders, missing unit_price). This is intentional — clean Silver = trustworthy Gold.")

## Summary

| Layer | What it does | Key transformations |
|---|---|---|
| **Bronze** | Land raw data, add `_load_ts` and `_source` | None — capture everything |
| **Silver** | Enforce quality, standardize, compute derived columns | Null key reject, dedup, status whitelist, derived metrics |
| **Gold** | Aggregate to business grain, answer one question each | JOIN, GROUP BY, business-named columns only |

**Remember:**
- Bronze row count = source row count (no filtering)
- Silver row count ≤ Bronze (quality rejects remove rows)
- Gold row count < Silver (aggregation collapses rows)
- Rebuild Gold from Silver, not from Bronze (Silver is the quality gate)

## Cleanup

Drop all demo tables created by this notebook.

In [ ]:
bonus_tables = [
    f"{BRONZE}.bonus_b2_customers",
    f"{BRONZE}.bonus_b2_orders",
    f"{SILVER}.bonus_b2_customers",
    f"{SILVER}.bonus_b2_orders_clean",
    f"{GOLD}.bonus_b2_daily_revenue",
    f"{GOLD}.bonus_b2_customer_summary",
]

for table in bonus_tables:
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    print(f"[OK] Dropped: {table}")

print("\nAll bonus B2 demo tables removed.")